In [1]:
!pip install mindspore

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 816.0/816.0 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 92.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


In [1]:
import mindspore

In [8]:
import math
import numpy as np
import pandas as pd
import os
import math
import random
import codecs
from pathlib import Path

import mindspore
import mindspore.dataset as ds
import mindspore.nn as nn
from mindspore import Tensor
from mindspore import context
from mindspore.train.model import Model
from mindspore.nn.metrics import Accuracy
from mindspore.train.serialization import load_checkpoint, load_param_into_net
from mindspore.train.callback import ModelCheckpoint, CheckpointConfig, LossMonitor, TimeMonitor
from mindspore.ops import operations as ops

from easydict import EasyDict as edict


cfg = edict({
'name': 'movie review', 'pre_trained': False, 'num_classes': 2,
'batch_size': 64,
'epoch_size': 4, 'weight_decay': 3e-5, 'data_path': '.', 'device_target': 'CPU', 'device_id': 0,
'keep_checkpoint_max': 1,
'checkpoint_path': './ckpt/train_textcnn-4_149.ckpt', 'word_len': 51,
'vec_length': 40
})
context.set_context(mode=context.GRAPH_MODE, device_target=cfg.device_target, device_id=cfg.device_id)

[WARNING] ME(4978:133286623715328,MainProcess):2025-07-17-07:55:46.466.000 [mindspore/context.py:1402] For 'context.set_context', the parameter 'device_target' will be deprecated and removed in a future version. Please use the api mindspore.set_device() instead.
[WARNING] ME(4978:133286623715328,MainProcess):2025-07-17-07:55:46.468.000 [mindspore/context.py:1402] For 'context.set_context', the parameter 'device_id' will be deprecated and removed in a future version. Please use the api mindspore.set_device() instead.


In [4]:
with open("rt-polarity.neg", 'r', encoding='utf-8') as f:
    print("Negative reivews:")
    for i in range(5):
        print("[{0}]:{1}".format(i,f.readline()))
with open("rt-polarity.pos", 'r', encoding='utf-8') as f:
    print("Positive reivews:")
    for i in range(5):
        print("[{0}]:{1}".format(i,f.readline()))

Negative reivews:
[0]:simplistic , silly and tedious . 

[1]:it's so laddish and juvenile , only teenage boys could possibly find it funny . 

[2]:exploitative and largely devoid of the depth or sophistication that would make watching such a graphic treatment of the crimes bearable . 

[3]:[garbus] discards the potential for pathological study , exhuming instead , the skewed melodrama of the circumstantial situation . 

[4]:a visually flashy but narratively opaque and emotionally vapid exercise in style and mystification . 

Positive reivews:
[0]:the rock is destined to be the 21st century's new " conan " and that he's going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal . 

[1]:the gorgeously elaborate continuation of " the lord of the rings " trilogy is so huge that a column of words cannot adequately describe co-writer/director peter jackson's expanded vision of j . r . r . tolkien's middle-earth . 

[2]:effective but too-tepid biopic

In [6]:
import os
import math
import random
import numpy as np
from pathlib import Path
import mindspore.dataset as ds  # assuming you're using MindSpore datasets

# ==============================
# ✅ Generator Class
# ==============================
class Generator:
    def __init__(self, input_list):
        self.input_list = input_list

    def __getitem__(self, item):
        return (
            np.array(self.input_list[item][0], dtype=np.int32),
            np.array(self.input_list[item][1], dtype=np.int32)
        )

    def __len__(self):
        return len(self.input_list)


# ==============================
# ✅ MovieReview Class
# ==============================
class MovieReview:
    """
    Movie review dataset handler
    """

    def __init__(self, root_dir, maxlen, split):
        """
        :param root_dir: movie review data directory
        :param maxlen: maximum length of a sentence
        :param split: the training/validation ratio in the dataset
        """
        self.path = root_dir
        self.feelMap = {'neg': 0, 'pos': 1}
        self.files = []
        self.doConvert = False

        # ✅ Validate directory
        mypath = Path(self.path)
        if not mypath.exists() or not mypath.is_dir():
            print("Please check the root_dir!")
            raise ValueError

        # ✅ Find the files in the data directory
        for root, _, filenames in os.walk(self.path):
            for each in filenames:
                self.files.append(os.path.join(root, each))
            break

        # ✅ Check whether there are two files: .neg and .pos
        if len(self.files) != 2:
            print("There are {} files in the root_dir".format(len(self.files)))
            raise ValueError

        # ✅ Read data
        self.word_num = 0
        self.maxlen = float("-inf")
        self.minlen = float("inf")
        self.Pos = []
        self.Neg = []

        for filename in self.files:
            self.read_data(filename)

        # ✅ Convert to vectors and split the dataset
        self.text2vec(maxlen=maxlen)
        self.split_dataset(split=split)

    def read_data(self, filePath):
        with open(filePath, 'r') as f:
            for sentence in f.readlines():
                sentence = (
                    sentence.replace('\n', '')
                    .replace('"', '')
                    .replace("'", '')
                    .replace('.', '')
                    .replace(',', '')
                    .replace('[', '')
                    .replace(']', '')
                    .replace('(', '')
                    .replace(')', '')
                    .replace(':', '')
                    .replace('--', '')
                    .replace('-', ' ')
                    .replace('\\', '')
                    .replace('0', '')
                    .replace('1', '')
                    .replace('2', '')
                    .replace('3', '')
                    .replace('4', '')
                    .replace('5', '')
                    .replace('6', '')
                    .replace('7', '')
                    .replace('8', '')
                    .replace('9', '')
                    .replace('`', '')
                    .replace('=', '')
                    .replace('$', '')
                    .replace('/', '')
                    .replace('*', '')
                    .replace(';', '')
                    .replace('<b>', '')
                    .replace('%', '')
                )

                sentence = sentence.split(' ')
                sentence = list(filter(lambda x: x, sentence))

                if sentence:
                    self.word_num += len(sentence)
                    self.maxlen = max(self.maxlen, len(sentence))
                    self.minlen = min(self.minlen, len(sentence))

                    if 'pos' in filePath:
                        self.Pos.append([sentence, self.feelMap['pos']])
                    else:
                        self.Neg.append([sentence, self.feelMap['neg']])

    def text2vec(self, maxlen):
        """
        Convert sentences into vectors.
        """
        self.Vocab = {}

        for SentenceLabel in self.Pos + self.Neg:
            vector = [0] * maxlen
            for index, word in enumerate(SentenceLabel[0]):
                if index >= maxlen:
                    break
                if word not in self.Vocab:
                    self.Vocab[word] = len(self.Vocab)
                vector[index] = self.Vocab[word]
            SentenceLabel[0] = vector

        self.doConvert = True

    def split_dataset(self, split):
        """
        Divide the dataset into a training set and a test set.
        """
        trunk_pos_size = math.ceil((1 - split) * len(self.Pos))
        trunk_neg_size = math.ceil((1 - split) * len(self.Neg))
        trunk_num = int(1 / (1 - split))

        pos_temp, neg_temp = [], []
        for index in range(trunk_num):
            pos_temp.append(self.Pos[index * trunk_pos_size:(index + 1) * trunk_pos_size])
            neg_temp.append(self.Neg[index * trunk_neg_size:(index + 1) * trunk_neg_size])

        self.test = pos_temp.pop(2) + neg_temp.pop(2)
        self.train = [i for item in pos_temp + neg_temp for i in item]
        random.shuffle(self.train)

    def get_dict_len(self):
        """
        Obtain the length of a dictionary consisting of characters in a dataset.
        """
        if self.doConvert:
            return len(self.Vocab)
        else:
            print("Haven't finished Text2Vec")
            return -1

    def create_train_dataset(self, epoch_size, batch_size):
        dataset = ds.GeneratorDataset(
            source=Generator(input_list=self.train),
            column_names=["data", "label"],
            shuffle=False
        )
        dataset = dataset.batch(batch_size=batch_size, drop_remainder=True)
        dataset = dataset.repeat(epoch_size)
        return dataset

    def create_test_dataset(self, batch_size):
        dataset = ds.GeneratorDataset(
            source=Generator(input_list=self.test),
            column_names=["data", "label"],
            shuffle=False
        )
        dataset = dataset.batch(batch_size=batch_size, drop_remainder=True)
        return dataset


In [9]:
instance = MovieReview(root_dir=cfg.data_path, maxlen=cfg.word_len, split=0.9)
dataset = instance.create_train_dataset(batch_size=cfg.batch_size,epoch_size=cfg.epoch_size)
batch_num = dataset.get_dataset_size()

In [12]:
vocab_size=instance.get_dict_len()
print("vocab_size:{0}".format(vocab_size))
item =dataset.create_dict_iterator()
for i,data in enumerate(item):
    if i<1:
      print(data)
      print(data['data'][1])
    else:
      break

vocab_size:18848
{'data': Tensor(shape=[64, 51], dtype=Int32, value=
[[ 2750,    15,    86 ...     0,     0,     0],
 [ 2524,  6423,   631 ...     0,     0,     0],
 [ 7904, 12618,  6706 ...     0,     0,     0],
 ...
 [    0,   108,  2844 ...     0,     0,     0],
 [   82,  6017,    60 ...     0,     0,     0],
 [   15,  2578,   146 ...     0,     0,     0]]), 'label': Tensor(shape=[64], dtype=Int32, value= [0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 
 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 
 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1])}
[2524 6423  631   64 6729    4 3079    0  287 2049 2011  122   15  345
   15   86 8911  253    0 8912   11  505  797   15  798    4 2418    4
 1863    4 1544    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0]


## 3. Configure training parameters.

In [13]:
learning_rate = []
warm_up = [1e-3 / math.floor(cfg.epoch_size / 5) * (i + 1) for _ in range(batch_num)
for i in range(math.floor(cfg.epoch_size / 5))]
shrink = [1e-3 / (16 * (i + 1)) for _ in range(batch_num)
for i in range(math.floor(cfg.epoch_size * 3 / 5))]
normal_run = [1e-3 for _ in range(batch_num)
for i in range(cfg.epoch_size - math.floor(cfg.epoch_size / 5) - math.floor(cfg.epoch_size * 2 / 5))]
learning_rate = learning_rate + warm_up + normal_run + shrink


The dynamic learning rate in training is implemented through epoch.

## 4. Define a TextCNN model.

Model classes define model structure build, training, validation, offline model loading, and online inference functions.

In [15]:
def _weight_variable(shape, factor=0.01):
  init_value = np.random.randn(*shape).astype(np.float32) * factor
  return Tensor(init_value)

def make_conv_layer(kernel_size):
  weight_shape = (96, 1, *kernel_size)
  weight = _weight_variable(weight_shape)
  return nn.Conv2d(in_channels=1, out_channels=96, kernel_size=kernel_size, padding=1, pad_mode="pad", weight_init=weight, has_bias=True)

class TextCNN(nn.Cell):
    def __init__(self, vocab_len, word_len, num_classes, vec_length):
        super(TextCNN, self).__init__()
        self.vec_length = vec_length
        self.word_len = word_len
        self.num_classes = num_classes

        self.unsqueeze = ops.ExpandDims()
        self.embedding = nn.Embedding(vocab_len, self.vec_length, embedding_table='normal')

        self.slice = ops.Slice()
        self.layer1 = self.make_layer(kernel_height=3)
        self.layer2 = self.make_layer(kernel_height=4)
        self.layer3 = self.make_layer(kernel_height=5)

        self.concat = ops.Concat(1)

        self.fc = nn.Dense(96*3, self.num_classes)
        self.drop = nn.Dropout(keep_prob=0.5)
        self.print = ops.Print()
        self.reducemean = ops.ReduceMax(keep_dims=False)

    def make_layer(self, kernel_height):
        return nn.SequentialCell(
            [
                make_conv_layer((kernel_height,self.vec_length)),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=(self.word_len-kernel_height+1,1)),
            ]
        )

    def construct(self,x):
        x = self.unsqueeze(x, 1)
        x = self.embedding(x)
        x1 = self.layer1(x)
        x2 = self.layer2(x)
        x3 = self.layer3(x)

        x1 = self.reducemean(x1, (2, 3))
        x2 = self.reducemean(x2, (2, 3))
        x3 = self.reducemean(x3, (2, 3))

        x = self.concat((x1, x2, x3))
        x = self.drop(x)
        x = self.fc(x)
        return x

net = TextCNN(vocab_len=instance.get_dict_len(), word_len=cfg.word_len, num_classes=cfg.num_classes, vec_length=cfg.vec_length)
print(net)

[WARNING] ME(4978:133286623715328,MainProcess):2025-07-17-08:05:31.314.000 [mindspore/nn/layer/basic.py:176] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.
[WARNING] ME(4978:133286623715328,MainProcess):2025-07-17-08:05:31.338.000 [mindspore/nn/layer/basic.py:202] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.


TextCNN(
  (embedding): Embedding(vocab_size=18848, embedding_size=40, use_one_hot=False, embedding_table=Parameter (name=embedding.embedding_table, shape=(18848, 40), dtype=Float32, requires_grad=True), dtype=Float32, padding_idx=None)
  (layer1): SequentialCell(
    (0): Conv2d(
      input_channels=1, output_channels=96, kernel_size=(3, 40), stride=(1, 1), pad_mode=pad, padding=1, dilation=(1, 1), group=1, has_bias=True, weight_init=[[[[-0.00276063 -0.01115905  0.0082776  ... -0.00825439 -0.00773969
           0.01365344]
         [-0.01089201 -0.0031637   0.00326144 ...  0.00544381  0.00747612
          -0.00672934]
         [ 0.00516619 -0.01337766 -0.01245807 ... -0.0064878   0.00944605
          -0.00774519]]]
      
      
       [[[-0.00113311  0.00039686 -0.01379627 ... -0.01443574 -0.00222839
           0.00902302]
         [-0.00691065 -0.00586954 -0.01293208 ... -0.01390252  0.00511619
          -0.01747015]
         [ 0.0051423   0.00777821 -0.00192092 ...  0.01439535 -0.

## 5. Define training parameters.

In [18]:
# Optimizer, loss function, checkpoint, and time monitor settings
opt = nn.Adam(filter(lambda x: x.requires_grad, net.get_parameters()), learning_rate=learning_rate, weight_decay=cfg.weight_decay)
loss = nn.SoftmaxCrossEntropyWithLogits(sparse=True)
model = Model(net, loss_fn=loss, optimizer=opt, metrics={'acc': Accuracy()})
config_ck = CheckpointConfig(save_checkpoint_steps=int(cfg.epoch_size*batch_num/2), keep_checkpoint_max=cfg.keep_checkpoint_max)
time_cb = TimeMonitor(data_size=batch_num)
ckpt_save_dir = "./ckpt"
ckpoint_cb = ModelCheckpoint(prefix="train_textcnn", directory=ckpt_save_dir, config=config_ck)
loss_cb = LossMonitor()


##6. Start training.

In [19]:
model.train(cfg.epoch_size, dataset, callbacks=[time_cb, ckpoint_cb, loss_cb])
print("train success")

[WARNING] ME(4978:133286623715328,MainProcess):2025-07-17-08:08:55.799.000 [mindspore/nn/layer/basic.py:202] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.


epoch: 1 step: 1, loss is 0.6909891366958618
epoch: 1 step: 2, loss is 0.6944298148155212
epoch: 1 step: 3, loss is 0.6925786733627319
epoch: 1 step: 4, loss is 0.6945385336875916
epoch: 1 step: 5, loss is 0.6932599544525146
epoch: 1 step: 6, loss is 0.6870759725570679
epoch: 1 step: 7, loss is 0.6930743455886841
epoch: 1 step: 8, loss is 0.6960666179656982
epoch: 1 step: 9, loss is 0.6992708444595337
epoch: 1 step: 10, loss is 0.696026086807251
epoch: 1 step: 11, loss is 0.684476375579834
epoch: 1 step: 12, loss is 0.6975305080413818
epoch: 1 step: 13, loss is 0.7003114223480225
epoch: 1 step: 14, loss is 0.6974570155143738
epoch: 1 step: 15, loss is 0.6908315420150757
epoch: 1 step: 16, loss is 0.699276328086853
epoch: 1 step: 17, loss is 0.6926717758178711
epoch: 1 step: 18, loss is 0.6942117810249329
epoch: 1 step: 19, loss is 0.6946103572845459
epoch: 1 step: 20, loss is 0.6925594806671143
epoch: 1 step: 21, loss is 0.6932508945465088
epoch: 1 step: 22, loss is 0.6914882659912109


##7. Test and validate data.

In [22]:
def preprocess(review_en):
  review_en = review_en.lower().strip()
  review_en = review_en.replace('\n','')\
                                                .replace('"','')\
                                                .replace('\'','')\
                                                .replace('.','')\
                                                .replace(',','')\
                                                .replace('[','')\
                                                .replace(']','')\
                                                .replace('(','')\
                                                .replace(')','')\
                                                .replace(':','')\
                                                .replace('--','')\
                                                .replace('-',' ')\
                                                .replace('\\','')\
                                                .replace('0','')\
                                                .replace('1','')\
                                                .replace('2','')\
                                                .replace('3','')\
                                                .replace('4','')\
                                                .replace('5','')\
                                                .replace('6','')\
                                                .replace('7','')\
                                                .replace('8','')\
                                                .replace('9','')\
                                                .replace('`','')\
                                                .replace('=','')\
                                                .replace('$','')\
                                                .replace('/','')\
                                                .replace('*','')\
      .replace(';','')\
                                                .replace('<b>','')\
                                                .replace('%','')\
                                                .replace("  "," ")
  sentence = review_en.split(' ')
  maxlen = cfg.word_len
  vector = [0]*maxlen
  for index, word in enumerate(sentence):
    if index >= maxlen:
      break
    if word not in instance.Vocab.keys():
      print(word,"The word does not appear in the dictionary.")
    else:
      vector[index] = instance.Vocab[word]
  sentence = vector
  return sentence

def inference(review_en):
  review_en = preprocess(review_en)
  input_en = Tensor(np.array([review_en]).astype(np.int32))
  output = net(input_en)
  if np.argmax(np.array(output[0])) == 1:
    print("Positive comments")
  else:
    print("Negative comments")

### Testing our Model using a sample review

In [23]:
review_en = "the movie is so boring"
inference(review_en)

[WARNING] ME(4978:133286623715328,MainProcess):2025-07-17-08:22:05.380.00 [mindspore/nn/layer/basic.py:202] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.


Negative comments


In [25]:
my_review = "I really enjoyed this movie!"
inference(my_review)

movie! The word does not appear in the dictionary.
Positive comments
